# 02 — Ingestion: CSV / Excel → SQLite

Pipeline A (structured data): validate a file, read each sheet as raw cells, clean column names, infer types, and import into SQLite with a catalog that records where every table and column came from.

Each piece is a **function in its own cell**, so it can move into `src/` later almost unchanged. Data exploration is in `01_EDA.ipynb`.

| Section | Builds |
|---|---|
| 0 | Setup |
| 2 | `validate_file()`, `list_sheets()` |
| 3 | `read_sheet()` — raw cells, header row, blank rows/columns |
| 4 | `normalize_column_names()` |
| 5 | `infer_column_type()` |
| 6 | Write a table to SQLite |
| 7 | Catalog tables |
| 8 | `import_file()` |
| 9 | Checks |

## Section 0 — Setup

In [ ]:
import json
import math
import re
import sqlite3
import unicodedata
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import date, datetime, timezone
from itertools import islice
from pathlib import Path

import numpy as np
import pandas as pd

In [101]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TEST_DIR = PROJECT_ROOT / "data" / "test"
DB_PATH = PROJECT_ROOT / "data" / "processed" / "chatbot.db"
MAX_FILE_SIZE_MB = 100

raw_file = RAW_DIR / "Womens Clothing E-Commerce Reviews.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("raw_file exists:", raw_file.exists())

PROJECT_ROOT: d:\ChatBot test
raw_file exists: True


## Section 2 — Validate files and list sheets
Check a file **before** reading it (exists, supported type, not too large), then list its sheets so the user can choose what to import. A CSV counts as one sheet named after the file.

`IngestionError` is the error for problems the *user* can fix; the importer will catch it and show a friendly message. Any other exception is a real bug and should crash loudly.

In [102]:
SUPPORTED_EXTENSIONS = (".xlsx", ".xls", ".csv")


class IngestionError(Exception):
    """A problem the user can fix: wrong file, unsupported layout, name conflict, etc."""

In [103]:
def validate_file(path, max_size_mb=MAX_FILE_SIZE_MB):
    """Check the file is importable and return it as a Path. Raises IngestionError otherwise."""
    path = Path(path)

    if path.name.startswith("~$"):
        raise IngestionError(
            f"'{path.name}' is an Excel lock file, not a workbook. Close Excel and import the real file."
        )
    if not path.is_file():
        raise IngestionError(f"File not found: {path}")
    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise IngestionError(
            f"Unsupported file type '{path.suffix}'. Supported: {', '.join(SUPPORTED_EXTENSIONS)}"
        )

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > max_size_mb:
        raise IngestionError(
            f"'{path.name}' is {size_mb:.1f} MB, above the {max_size_mb} MB limit."
        )
    return path

In [104]:
def list_sheets(path):
    """Sheet names in a workbook. A CSV counts as one sheet named after the file."""
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return [path.stem]
    try:
        with pd.ExcelFile(path) as workbook:
            return [str(name) for name in workbook.sheet_names]
    except Exception as exc:
        raise IngestionError(f"Could not open '{path.name}': {exc}") from exc

### Check: a valid file passes, bad files are rejected
Expected: the reviews CSV lists one sheet, `Womens Clothing E-Commerce Reviews`; the second cell prints four "Rejected as expected" lines and no `UNEXPECTED`.

In [105]:
good = validate_file(raw_file)
print("OK:", good)
print("Sheets:", list_sheets(good))

OK: d:\ChatBot test\data\raw\Womens Clothing E-Commerce Reviews.csv
Sheets: ['Womens Clothing E-Commerce Reviews']


In [106]:
bad_cases = [
    PROJECT_ROOT / "does_not_exist.xlsx",                     # missing file
    PROJECT_ROOT / "excel_query_answering_chatbot_spec.md",   # wrong type
    PROJECT_ROOT / "~$Online Retail.xlsx",                    # Excel lock file
]
for case in bad_cases:
    try:
        validate_file(case)
        print("UNEXPECTED: no error for", case.name)
    except IngestionError as exc:
        print("Rejected as expected ->", exc)

try:
    validate_file(raw_file, max_size_mb=1)
    print("UNEXPECTED: no error")
except IngestionError as exc:
    print("Rejected as expected ->", exc)

Rejected as expected -> File not found: d:\ChatBot test\does_not_exist.xlsx
Rejected as expected -> Unsupported file type '.md'. Supported: .xlsx, .xls, .csv
Rejected as expected -> '~$Online Retail.xlsx' is an Excel lock file, not a workbook. Close Excel and import the real file.
Rejected as expected -> 'Womens Clothing E-Commerce Reviews.csv' is 8.1 MB, above the 1 MB limit.


## Section 3 — Read a sheet as raw cells
`read_sheet()` reads one sheet **without guessing types** (that is Section 5), then:
1. treats **row 1 as the header** and refuses sheets whose row 1 is empty;
2. **cleans cells**: trims spaces; turns `""` and Excel errors like `#N/A` into `None`;
3. **drops** columns with no header *and* no values, and completely blank rows — and **reports** both;
4. keeps each row's **source row number** as the index (header = row 1) so answers can cite "row 42".

`RawSheet` bundles the result. `field(default_factory=list)` gives each object its own warnings list (a plain `= []` would be shared by all objects).

In [107]:
CSV_ENCODINGS = ("utf-8-sig", "cp1252")
EXCEL_ERROR_VALUES = {"#N/A", "#DIV/0!", "#VALUE!", "#REF!", "#NAME?", "#NUM!", "#NULL!"}


@dataclass
class RawSheet:
    """One sheet as read from disk: header texts + cleaned, untyped cells."""
    source_path: Path
    sheet_name: str
    headers: list          # header text per column ("" if blank)
    data: pd.DataFrame     # columns 0..n-1, index = source row number
    warnings: list = field(default_factory=list)

### Read every cell exactly as stored
Each option switches off a pandas "helpful" behaviour that would hide or change data:

| Option | Why |
|---|---|
| `header=None` | Otherwise pandas renames a blank header to `Unnamed: 0` and a duplicate `Name` to `Name.1`. We want the real headers. |
| `dtype=object` | Keep values as stored; no guessing (e.g. IDs turning into floats). |
| `keep_default_na=False, na_values=[]` | Stop pandas turning the *text* `"NA"`/`"null"` into missing values. |
| `skip_blank_lines=False` | Keep blank CSV lines so row numbers still match the file. |
| two encodings | Windows Excel exports are often `cp1252`, not UTF-8. |

In [108]:
def read_raw_cells(path, sheet_name):
    """Read every cell exactly as stored: no header detection, no type conversion."""
    path = Path(path)
    if path.suffix.lower() == ".csv":
        for encoding in CSV_ENCODINGS:
            try:
                return pd.read_csv(path, header=None, dtype=object, keep_default_na=False,
                                   na_values=[], skip_blank_lines=False, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise IngestionError(f"Could not decode '{path.name}' as UTF-8 or Windows-1252 text")
    return pd.read_excel(path, sheet_name=sheet_name, header=None, dtype=object,
                         keep_default_na=False, na_values=[])

### Clean the cells
Every kind of blank (`""`, `"   "`, `NaN`, `NaT`, `pd.NA`, Excel errors) becomes **one** value, `None`, so later code checks for just one thing.

Note: in `.xlsx` files pandas already turns Excel error cells into blanks before we see them, so the error count only appears for CSV files.

In [109]:
def clean_cell(value):
    """Trim text; turn empty text, Excel errors and NaN into None; keep everything else."""
    if isinstance(value, str):
        value = value.strip()
        if value == "" or value in EXCEL_ERROR_VALUES:
            return None
        return value
    if value is None or value is pd.NaT or value is pd.NA:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def clean_cells(frame):
    """Apply clean_cell to every cell. Returns (cleaned frame, number of Excel error values)."""
    error_cells = int(frame.isin(EXCEL_ERROR_VALUES).sum().sum())
    cleaned = {
        col: pd.Series([clean_cell(v) for v in frame[col]], index=frame.index, dtype=object)
        for col in frame.columns
    }
    return pd.DataFrame(cleaned, index=frame.index), error_cells

### Put it together: `read_sheet()`
- **Order matters:** clean first, then look for blanks — a row of `"   "` only counts as blank after `strip()`.
- `except IngestionError: raise` stops the general `except Exception` from wrapping our own friendly error a second time.
- A column with a **blank header but real values** is kept (Section 4 names it). The reviews file has exactly this: its first column.

In [110]:
def header_text(value):
    """Header cell as text: None -> "", 2020.0 -> "2020"."""
    if value is None:
        return ""
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    return str(value).strip()


def read_sheet(path, sheet_name):
    path = Path(path)
    try:
        raw = read_raw_cells(path, sheet_name)
    except IngestionError:
        raise
    except Exception as exc:
        raise IngestionError(f"Could not read sheet '{sheet_name}' from '{path.name}': {exc}") from exc

    raw.index = range(1, len(raw) + 1)          # row numbers as in Excel: header = row 1
    raw, error_cells = clean_cells(raw)

    if raw.empty or raw.iloc[0].isna().all():
        raise IngestionError(
            f"Sheet '{sheet_name}' has no header in row 1. This version needs a single header row in the first row."
        )

    warnings = []
    if error_cells:
        warnings.append(f"Treated {error_cells} Excel error value(s) such as #N/A as blank")

    headers = [header_text(v) for v in raw.iloc[0]]
    data = raw.iloc[1:]

    # Drop columns that have neither a header nor any values
    keep = [col for col, h in zip(raw.columns, headers) if h or data[col].notna().any()]
    if len(keep) < len(raw.columns):
        warnings.append(f"Dropped {len(raw.columns) - len(keep)} empty column(s) with no header")
    headers = [h for col, h in zip(raw.columns, headers) if col in keep]
    data = data[keep]

    # Drop rows where every cell is blank, and say which ones
    blank = data.isna().all(axis=1)
    if blank.any():
        rows = list(data.index[blank])
        warnings.append(f"Skipped {len(rows)} blank row(s): source rows {rows[:10]}")
        data = data[~blank]
    if data.empty:
        warnings.append("Sheet has a header row but no data rows")

    data.columns = range(len(headers))
    return RawSheet(path, sheet_name, headers, data, warnings)

### Check 1: a small messy workbook
The reviews CSV is clean, so we build a deliberately messy `.xlsx` to prove the edge cases work: duplicate and blank headers, extra spaces, an Excel error, a row of spaces, and the text `"NA"`.

Expected:
```
Headers: ['Name', 'Name', '', 'Amount', 'Notes']
Warnings: ['Dropped 1 empty column(s) with no header', 'Skipped 1 blank row(s): source rows [4]']
```
and a table indexed **2, 3, 5** where `hello` is trimmed, `#DIV/0!` and `""` are `None`, and `NA` is still text.

In [111]:
from openpyxl import Workbook

TEST_DIR.mkdir(parents=True, exist_ok=True)
messy_file = TEST_DIR / "messy.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Messy Data"
ws.append(["Name", "Name", None, "Amount", "Notes", None])   # duplicate + blank headers
ws.append(["Alice", "A", "x1", 10, "  hello  ", None])       # extra spaces
ws.append(["Bob", "B", "x2", "#DIV/0!", "", None])           # Excel error + empty text
ws.append(["   ", None, None, None, None, "  "])             # only spaces -> blank row
ws.append(["Cara", "C", "x3", 7.5, "NA", None])              # "NA" must stay text
wb.save(messy_file)

In [112]:
messy = read_sheet(messy_file, "Messy Data")
print("Headers:", messy.headers)
print("Warnings:", messy.warnings)
messy.data

Headers: ['Name', 'Name', '', 'Amount', 'Notes']
Warnings: ['Dropped 1 empty column(s) with no header', 'Skipped 1 blank row(s): source rows [4]']


,0,1,2,3,4
2,Alice,A,x1,10,hello
3,Bob,B,x2,None,None
5,Cara,C,x3,7.5,NA


### Check 2: a tiny CSV and a sheet with no header
Expected: the CSV skips blank row 3 (index `2, 4`) and keeps `"NA"` as text; the no-header workbook prints one "Rejected as expected" line.

In [113]:
tiny_csv = TEST_DIR / "tiny.csv"
tiny_csv.write_text("Region,Sales\nNorth,10\n,\nNA,5\n")
tiny = read_sheet(tiny_csv, "tiny")
print(tiny.headers, tiny.warnings)
print(tiny.data, "\n")

no_header = TEST_DIR / "no_header.xlsx"
wb = Workbook()
wb.active.append([])        # row 1 empty
wb.active.append(["x"])
wb.save(no_header)
try:
    read_sheet(no_header, "Sheet")
    print("UNEXPECTED: no error")
except IngestionError as exc:
    print("Rejected as expected ->", exc)

['Region', 'Sales'] ['Skipped 1 blank row(s): source rows [3]']
       0   1
2  North  10
4     NA   5 

Rejected as expected -> Sheet 'Sheet' has no header in row 1. This version needs a single header row in the first row.


### Check 3: the real reviews file
Expected: **23,486 rows, 11 headers, no warnings**. The first header is `''` (blank) — that is the leftover index column found in `01_EDA`; Section 4 will name it and we will rename it `review_id`.

Every value is still a **string** (`str`) because nothing is typed yet — Section 5 turns `"4"` into the integer 4.

In [114]:
reviews_sheet = read_sheet(raw_file, list_sheets(raw_file)[0])
print("Headers:", reviews_sheet.headers)
print("Rows:", len(reviews_sheet.data), "| Warnings:", reviews_sheet.warnings)
print("Index starts at:", reviews_sheet.data.index[0])
reviews_sheet.data.head(3)

Headers: ['', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']
Rows: 23486 | Warnings: []
Index starts at: 2


,0,1,2,3,4,5,6,7,8,9,10
2,0,767,33,None,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
3,1,1080,34,None,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
4,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses


In [115]:
# Type of each value in the Rating column (column 5) - all text for now
reviews_sheet.data[5].map(type).value_counts()

5
<class 'str'>    23486
Name: count, dtype: int64

## Section 4 — Clean column names
Headers become lowercase snake_case SQL names. Original headers are kept for the catalog.
Blank → `column_N`, duplicates → `_2`, leading digit → `c_`, SQL keywords → `_col`.


In [117]:
MAX_IDENTIFIER_LENGTH = 63   # PostgreSQL's limit, so names stay portable

# SQL keywords: a column named "order" would break generated SQL unless quoted
RESERVED_WORDS = {
    "all", "and", "as", "asc", "between", "by", "case", "check", "column", "create", "default",
    "delete", "desc", "distinct", "drop", "else", "end", "exists", "from", "group", "having", "in",
    "index", "insert", "is", "join", "key", "like", "limit", "not", "null", "offset", "on", "or",
    "order", "primary", "select", "table", "then", "to", "union", "update", "values", "when", "where",
}


def normalize_name(name, fallback):
    """'Unit Price ($)' -> 'unit_price'. Returns fallback if nothing usable is left."""
    text = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^0-9A-Za-z]+", "_", text).strip("_").lower()
    if not text:
        return fallback
    if text[0].isdigit():
        text = f"c_{text}"
    if text in RESERVED_WORDS:
        text = f"{text}_col"
    return text[:MAX_IDENTIFIER_LENGTH].rstrip("_")


In [118]:
def dedupe_names(names):
    """['name', 'name'] -> ['name', 'name_2']"""
    used = set()
    result = []
    for name in names:
        candidate, suffix = name, 2
        while candidate in used:
            candidate = f"{name}_{suffix}"
            suffix += 1
        used.add(candidate)
        result.append(candidate)
    return result


def normalize_column_names(headers):
    return dedupe_names(
        [normalize_name(h, fallback=f"column_{i}") for i, h in enumerate(headers, start=1)]
    )


In [119]:
def apply_renames(names, renames):
    """Apply user-chosen renames, e.g. {"column_1": "review_id"}; refuse unknown or clashing names."""
    unknown = [old for old in renames if old not in names]
    if unknown:
        raise IngestionError(f"Cannot rename {unknown}: no such column. Columns are: {names}")
    renamed = [renames.get(n, n) for n in names]
    clashes = sorted({n for n in renamed if renamed.count(n) > 1})
    if clashes:
        raise IngestionError(f"Renaming would create duplicate column names: {clashes}")
    return renamed


COLUMN_RENAMES = {"column_1": "review_id"}


### Check: name cleaning

In [120]:
cases = {
    "Invoice No": "invoice_no",
    "Unit Price ($)": "unit_price",
    "2020 Sales": "c_2020_sales",
    "Order": "order_col",
    "Café Total": "cafe_total",
    "  Review   Text ": "review_text",
    "": "column_9",
}
for raw, expected in cases.items():
    got = normalize_name(raw, fallback="column_9")
    assert got == expected, f"{raw!r}: expected {expected!r}, got {got!r}"
print("normalize_name: all cases pass")

print("Messy:", normalize_column_names(messy.headers))

review_columns = apply_renames(normalize_column_names(reviews_sheet.headers), COLUMN_RENAMES)
print("Reviews:", review_columns)

original_columns = normalize_column_names(reviews_sheet.headers)   # before renaming
for bad in [{"column_99": "x"}, {"column_1": "age"}]:
    try:
        apply_renames(original_columns, bad)
        print("UNEXPECTED: no error for", bad)
    except IngestionError as exc:
        print("Rejected as expected ->", exc)



normalize_name: all cases pass
Messy: ['name', 'name_2', 'column_3', 'amount', 'notes']
Reviews: ['review_id', 'clothing_id', 'age', 'title', 'review_text', 'rating', 'recommended_ind', 'positive_feedback_count', 'division_name', 'department_name', 'class_name']
Rejected as expected -> Cannot rename ['column_99']: no such column. Columns are: ['column_1', 'clothing_id', 'age', 'title', 'review_text', 'rating', 'recommended_ind', 'positive_feedback_count', 'division_name', 'department_name', 'class_name']
Rejected as expected -> Renaming would create duplicate column names: ['age']


## Section 5 — Infer column types
Rule: a column gets a non-text type only if **every** non-blank value converts; otherwise it stays text with a warning.
Order of checks: empty → boolean → real dates → numbers → date-like text → text.

In [122]:
MIXED_TYPE_WARNING_RATIO = 0.9    # warn when a text column is "almost" numeric
DATETIME_SAMPLE_SIZE = 1000       # try date parsing on a sample first (it's slow)
SAMPLE_VALUE_COUNT = 3
DATE_FORMATS = {"date": "%Y-%m-%d", "datetime": "%Y-%m-%d %H:%M:%S"}
BOOLEAN_STRINGS = {"true": True, "false": False, "yes": True, "no": False}
LEADING_ZERO = re.compile(r"^[-+]?0\d")
DATE_LIKE = re.compile(
    r"\d{1,4}[-/.]\d{1,2}"                      # 2024-01-15, 12/1/2010, 15.01.2024
    r"|\d{1,2}\s+[A-Za-z]{3,9}\.?\s+\d{2,4}"    # 1 Dec 2010
    r"|[A-Za-z]{3,9}\.?\s+\d{1,2},?\s+\d{2,4}"  # Dec 1, 2010
)


def is_bool_like(value):
    return isinstance(value, (bool, np.bool_)) or (isinstance(value, str) and value.lower() in BOOLEAN_STRINGS)


def to_bool(value):
    return BOOLEAN_STRINGS[value.lower()] if isinstance(value, str) else bool(value)


def is_datetime_like(value):
    # datetime and pandas Timestamp are both subclasses of date
    return isinstance(value, (date, np.datetime64))


def numeric_candidate(value):
    """Value to try as a number, or None. Booleans are excluded: True is not 1 here."""
    if isinstance(value, (bool, np.bool_)):
        return None
    if isinstance(value, (int, float, np.integer, np.floating, str)):
        return value
    return None


def to_text(value):
    """Text form of a value: 17850.0 -> '17850', datetime -> '2020-01-01 10:30:00'."""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    if isinstance(value, datetime):
        return value.isoformat(sep=" ")
    return str(value)


In [123]:
def datetime_result(parsed, warnings):
    """Decide date vs datetime; convert time zones to UTC."""
    if parsed.dt.tz is not None:
        parsed = parsed.dt.tz_convert("UTC").dt.tz_localize(None)
        warnings = warnings + ["time zones converted to UTC"]
    non_null = parsed.dropna()
    data_type = "date" if (non_null == non_null.dt.normalize()).all() else "datetime"
    return data_type, parsed, warnings


def parse_date_strings(values, items):
    """Try to read text as dates. Returns (parsed, format) or (None, None)."""
    sample = items[:DATETIME_SAMPLE_SIZE]
    if not all(is_datetime_like(v) or (isinstance(v, str) and DATE_LIKE.search(v)) for v in sample):
        return None, None
    as_text = values.map(lambda v: v.isoformat() if is_datetime_like(v) else v, na_action="ignore")
    for fmt in ("ISO8601", "mixed"):
        try:
            if pd.to_datetime(as_text.dropna().head(DATETIME_SAMPLE_SIZE), errors="coerce", format=fmt).isna().any():
                continue
            parsed = pd.to_datetime(as_text, errors="coerce", format=fmt)
        except (ValueError, TypeError):
            continue
        if parsed.notna().sum() == len(items):
            return parsed, fmt
    return None, None


In [124]:
def infer_column_type(values):
    """Return (data_type, typed values, warnings) for one column of cleaned cells."""
    items = values.dropna().tolist()
    if not items:
        return "text", values.astype(object), ["column has no values"]

    if all(is_bool_like(v) for v in items):
        return "boolean", values.map(to_bool, na_action="ignore").astype("boolean"), []

    if all(is_datetime_like(v) for v in items):
        return datetime_result(pd.to_datetime(values), [])

    numbers = pd.to_numeric(values.map(numeric_candidate).astype(object), errors="coerce")
    numeric_ratio = numbers.notna().sum() / len(items)
    leading_zero = next((v for v in items if isinstance(v, str) and LEADING_ZERO.match(v)), None)

    if numeric_ratio == 1 and leading_zero is None:
        present = numbers.dropna()
        if (present % 1 == 0).all() and present.abs().max() < 2**63:
            return "integer", numbers.astype("Int64"), []
        return "decimal", numbers.astype("Float64"), []

    if numeric_ratio < 1:
        parsed, fmt = parse_date_strings(values, items)
        if parsed is not None:
            warnings = []
            if fmt == "mixed":
                warnings.append("dates parsed with a flexible format; ambiguous values like 01/02/2020 were read month-first")
            return datetime_result(parsed, warnings)

    warnings = []
    if leading_zero is not None and numeric_ratio == 1:
        warnings.append(f"stored as text to keep leading zeros (e.g. '{leading_zero}')")
    elif numeric_ratio >= MIXED_TYPE_WARNING_RATIO:
        non_numeric = values[values.notna() & numbers.isna()]
        examples = ", ".join(f"'{v}'" for v in non_numeric.drop_duplicates().head(3))
        warnings.append(
            f"stored as text: {numeric_ratio:.1%} of values are numbers but {len(non_numeric)} are not (e.g. {examples})"
        )
    return "text", values.map(to_text, na_action="ignore").astype(object), warnings


In [125]:
def to_storage_values(typed, data_type):
    """Typed values -> plain Python values SQLite accepts (None for blanks)."""
    if data_type in DATE_FORMATS:
        typed = typed.dt.strftime(DATE_FORMATS[data_type])
    cast = {"integer": int, "decimal": float, "boolean": int}.get(data_type, str)
    return [None if pd.isna(v) else cast(v) for v in typed.astype(object)]


@dataclass
class ColumnProfile:
    original_name: str
    name: str
    data_type: str
    null_count: int
    distinct_count: int
    sample_values: list
    min_value: str = None
    max_value: str = None
    avg_length: float = None       # text columns only
    warnings: list = field(default_factory=list)


def profile_column(values, original_name, name):
    """Infer the type and summarise the column. Returns (typed values, ColumnProfile)."""
    data_type, typed, warnings = infer_column_type(values)
    non_null = typed.dropna()
    samples = to_storage_values(non_null.drop_duplicates().head(SAMPLE_VALUE_COUNT), data_type)
    if data_type == "boolean":
        samples = ["true" if v else "false" for v in samples]
    profile = ColumnProfile(
        original_name, name, data_type,
        null_count=int(typed.isna().sum()),
        distinct_count=int(non_null.nunique()),
        sample_values=[str(v) for v in samples],
        warnings=warnings,
    )
    if len(non_null):
        if data_type in ("integer", "decimal", "date", "datetime"):
            low, high = to_storage_values(pd.Series([non_null.min(), non_null.max()]), data_type)
            profile.min_value, profile.max_value = str(low), str(high)
        elif data_type == "text":
            profile.avg_length = round(float(non_null.map(len).mean()), 1)
    return typed, profile


### Check: type inference

In [126]:
def col(*values):
    return pd.Series(list(values), dtype=object)

cases = [
    (col("1", "2", None), "integer"),
    (col(1, 2.5), "decimal"),
    (col(1.0, 2.0, None), "integer"),
    (col("007", "012"), "text"),
    (col(True, False, None), "boolean"),
    (col("Yes", "no"), "boolean"),
    (col("0", "1", "1"), "integer"),
    (col(datetime(2020, 1, 1, 10, 30), None), "datetime"),
    (col(datetime(2020, 1, 1), datetime(2020, 2, 1)), "date"),
    (col("2024-01-15", "2024-02-01"), "date"),
    (col("12/1/2010 8:26", "12/2/2010 9:00"), "datetime"),
    (col(None, None), "text"),
    (col("apple", "banana"), "text"),
    (col(536365, "C536379"), "text"),
]
for values, expected in cases:
    got, _, warnings = infer_column_type(values)
    assert got == expected, f"{list(values)}: expected {expected}, got {got}"
print(f"infer_column_type: all {len(cases)} cases pass")

almost_numeric = col(*[str(i) for i in range(19)], "X1")
print(infer_column_type(almost_numeric)[2])


infer_column_type: all 14 cases pass
["stored as text: 95.0% of values are numbers but 1 are not (e.g. 'X1')"]


In [127]:
review_profiles = []
for i, (header, name) in enumerate(zip(reviews_sheet.headers, review_columns)):
    typed, profile = profile_column(reviews_sheet.data[i], header, name)
    review_profiles.append(profile)

pd.DataFrame([
    {"name": p.name, "type": p.data_type, "nulls": p.null_count, "distinct": p.distinct_count,
     "min": p.min_value, "max": p.max_value, "avg_len": p.avg_length, "warnings": p.warnings}
    for p in review_profiles
])


,name,type,nulls,distinct,min,max,avg_len,warnings
0,review_id,integer,0,23486,0,23485,NaN,[]
1,clothing_id,integer,0,1206,0,1205,NaN,[]
2,age,integer,0,77,18,99,NaN,[]
3,title,text,3810,13993,NaN,NaN,19.1,[]
4,review_text,text,845,22634,NaN,NaN,308.7,[]
5,rating,integer,0,5,1,5,NaN,[]
6,recommended_ind,integer,0,2,0,1,NaN,[]
7,positive_feedback_count,integer,0,82,0,122,NaN,[]
8,division_name,text,14,3,NaN,NaN,9.5,[]
9,department_name,text,14,6,NaN,NaN,5.7,[]


## Section 6 — Write a table to SQLite
Each table gets `_row_id` (source row number) as primary key. Writes run inside a transaction:
if anything fails, SQLite rolls back and the previous table is untouched. Existing tables are
never overwritten unless `replace=True`. Later phases open the database **read-only**.


In [129]:
ROW_ID_COLUMN = "_row_id"
INSERT_BATCH_SIZE = 5_000

# Our logical types -> SQLite column types. SQLite has no date or boolean type.
SQLITE_TYPES = {
    "integer": "INTEGER",
    "decimal": "REAL",
    "boolean": "INTEGER",   # 1 / 0
    "date": "TEXT",         # 'YYYY-MM-DD'
    "datetime": "TEXT",     # 'YYYY-MM-DD HH:MM:SS'
    "text": "TEXT",
}


def quote_identifier(name):
    """Wrap a table/column name in double quotes so SQL treats it as a name, never as code."""
    return '"' + name.replace('"', '""') + '"'


def connect_db(path, read_only=False):
    path = Path(path)
    if read_only:
        if not path.is_file():
            raise FileNotFoundError(f"Database not found: {path}")
        # mode=ro: SQLite itself refuses every write on this connection
        conn = sqlite3.connect(f"{path.resolve().as_uri()}?mode=ro", uri=True)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        conn = sqlite3.connect(path)
    conn.isolation_level = None          # we start and end transactions ourselves
    conn.row_factory = sqlite3.Row       # rows can be read by column name
    return conn


In [130]:
@contextmanager
def transaction(conn):
    """with transaction(conn): ...  -> COMMIT if the block succeeds, ROLLBACK if it raises."""
    conn.execute("BEGIN")
    try:
        yield
    except BaseException:
        conn.execute("ROLLBACK")
        raise
    else:
        conn.execute("COMMIT")


In [131]:
def table_exists(conn, name):
    row = conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?", (name,)
    ).fetchone()
    return row is not None


def create_table(conn, name, columns, primary_key=None):
    """columns: list of (column_name, logical_type)."""
    definitions = []
    for column_name, data_type in columns:
        definition = f"{quote_identifier(column_name)} {SQLITE_TYPES[data_type]}"
        if column_name == primary_key:
            definition += " PRIMARY KEY"
        definitions.append(definition)
    conn.execute(f"CREATE TABLE {quote_identifier(name)} ({', '.join(definitions)})")


def insert_rows(conn, name, column_names, rows):
    """Insert rows in batches. Returns how many were inserted."""
    columns = ", ".join(quote_identifier(c) for c in column_names)
    placeholders = ", ".join("?" for _ in column_names)
    sql = f"INSERT INTO {quote_identifier(name)} ({columns}) VALUES ({placeholders})"
    inserted = 0
    row_iter = iter(rows)
    while batch := list(islice(row_iter, INSERT_BATCH_SIZE)):
        conn.executemany(sql, batch)
        inserted += len(batch)
    return inserted


In [132]:
def write_table(conn, name, columns, rows, replace=False):
    """Create the table and insert rows. Call inside `with transaction(conn):`."""
    if table_exists(conn, name):
        if not replace:
            raise IngestionError(f"Table '{name}' already exists. Use replace=True to overwrite it.")
        conn.execute(f"DROP TABLE {quote_identifier(name)}")
    create_table(conn, name, columns, primary_key=ROW_ID_COLUMN)
    inserted = insert_rows(conn, name, [c for c, _ in columns], rows)
    counted = conn.execute(f"SELECT COUNT(*) FROM {quote_identifier(name)}").fetchone()[0]
    if counted != inserted:
        raise IngestionError(f"Row count check failed for '{name}': inserted {inserted}, found {counted}")
    return counted


def query(conn, sql, params=()):
    """Run a SELECT and return the result as a DataFrame (for checks and exploring)."""
    return pd.DataFrame([dict(r) for r in conn.execute(sql, params).fetchall()])


### Check: write the reviews into a **test** database
We use `data/test/test.db`, not the real database — Section 8 does the real import, with the catalog.


In [133]:
review_stored = []
for i, profile in enumerate(review_profiles):
    typed, _ = profile_column(reviews_sheet.data[i], profile.original_name, profile.name)
    review_stored.append(to_storage_values(typed, profile.data_type))

review_table_columns = [(ROW_ID_COLUMN, "integer")] + [(p.name, p.data_type) for p in review_profiles]
review_rows = list(zip(reviews_sheet.data.index.tolist(), *review_stored))
print(len(review_table_columns), "columns,", len(review_rows), "rows")
print("First row:", review_rows[0][:7])

test_db = TEST_DIR / "test.db"
test_db.unlink(missing_ok=True)          # start fresh each run
conn = connect_db(test_db)
with transaction(conn):
    written = write_table(conn, "reviews", review_table_columns, review_rows)
print("Wrote", written, "rows")


12 columns, 23486 rows
First row: (2, 0, 767, 33, None, 'Absolutely wonderful - silky and sexy and comfortable', 4)
Wrote 23486 rows


In [134]:
# 1. Existing table without replace=True -> refused
try:
    with transaction(conn):
        write_table(conn, "reviews", review_table_columns, review_rows)
    print("UNEXPECTED: no error")
except IngestionError as exc:
    print("Rejected as expected ->", exc)

# 2. A replace that fails halfway -> rolled back, old table intact
bad_rows = review_rows[:10] + [(999999, "too", "few")]     # last row has 3 values, not 12
try:
    with transaction(conn):
        write_table(conn, "reviews", review_table_columns, bad_rows, replace=True)
    print("UNEXPECTED: no error")
except sqlite3.Error as exc:
    print("Failed as expected ->", type(exc).__name__, exc)
rows_now = conn.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
assert rows_now == 23486, rows_now
print("Rows after failed replace:", rows_now, "(old table survived)")


Rejected as expected -> Table 'reviews' already exists. Use replace=True to overwrite it.
Failed as expected -> ProgrammingError Incorrect number of bindings supplied. The current statement uses 12, and there are 3 supplied.
Rows after failed replace: 23486 (old table survived)


In [135]:
query(conn, """
    SELECT department_name,
           COUNT(*)              AS reviews,
           ROUND(AVG(rating), 2) AS avg_rating,
           SUM(rating <= 2)      AS negative_reviews
    FROM reviews
    WHERE department_name IS NOT NULL
    GROUP BY department_name
    ORDER BY negative_reviews DESC
""")


,department_name,reviews,avg_rating,negative_reviews
0,Tops,10468,4.17,1114
1,Dresses,6319,4.15,689
2,Bottoms,3799,4.29,320
3,Intimate,1735,4.28,153
4,Jackets,1032,4.26,109
5,Trend,119,3.82,22


In [136]:
conn.close()

ro = connect_db(test_db, read_only=True)
try:
    ro.execute("DELETE FROM reviews")
    print("UNEXPECTED: write allowed")
except sqlite3.OperationalError as exc:
    print("Read-only rejected the write ->", exc)
print("Rows still there:", ro.execute("SELECT COUNT(*) FROM reviews").fetchone()[0])
ro.close()


Read-only rejected the write -> attempt to write a readonly database
Rows still there: 23486


## Section 7 — Catalog
Two tables describe every imported table: `_catalog_tables` (source, row count, import time, warnings)
and `_catalog_columns` (original header, type, blanks, samples, min/max). Later notebooks build the
LLM's schema description from the catalog — the LLM never sees raw rows.


In [137]:
CATALOG_TABLES = "_catalog_tables"
CATALOG_COLUMNS = "_catalog_columns"
TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"
CATEGORY_LIMIT = 25     # text columns with this many distinct values or fewer are listed in full


def ensure_catalog(conn):
    """Create the two catalog tables if they don't exist yet."""
    conn.execute(f"""
        CREATE TABLE IF NOT EXISTS {CATALOG_TABLES} (
            table_name         TEXT PRIMARY KEY,
            source_file        TEXT,
            source_path        TEXT,
            source_sheet       TEXT,
            source_modified_at TEXT,
            imported_at        TEXT,
            row_count          INTEGER,
            warnings           TEXT
        )""")
    conn.execute(f"""
        CREATE TABLE IF NOT EXISTS {CATALOG_COLUMNS} (
            table_name     TEXT,
            position       INTEGER,
            column_name    TEXT,
            original_name  TEXT,
            data_type      TEXT,
            null_count     INTEGER,
            distinct_count INTEGER,
            sample_values  TEXT,
            min_value      TEXT,
            max_value      TEXT,
            avg_length     REAL,
            PRIMARY KEY (table_name, column_name)
        )""")


In [138]:
def utc_timestamp(value):
    return value.astimezone(timezone.utc).strftime(TIMESTAMP_FORMAT)


def record_import(conn, table_name, source_path, sheet_name, row_count, profiles, warnings):
    """Replace the catalog entries for one table. Call inside the same transaction as write_table."""
    source_path = Path(source_path)
    modified = datetime.fromtimestamp(source_path.stat().st_mtime, tz=timezone.utc)

    conn.execute(f"DELETE FROM {CATALOG_TABLES} WHERE table_name = ?", (table_name,))
    conn.execute(f"DELETE FROM {CATALOG_COLUMNS} WHERE table_name = ?", (table_name,))

    conn.execute(
        f"INSERT INTO {CATALOG_TABLES} VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
        (table_name, source_path.name, str(source_path.resolve()), sheet_name,
         utc_timestamp(modified), utc_timestamp(datetime.now(timezone.utc)),
         row_count, json.dumps(warnings)),
    )
    conn.executemany(
        f"INSERT INTO {CATALOG_COLUMNS} VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        [(table_name, position, p.name, p.original_name, p.data_type, p.null_count, p.distinct_count,
          json.dumps(p.sample_values), p.min_value, p.max_value, p.avg_length)
         for position, p in enumerate(profiles, start=1)],
    )


In [139]:
def catalog_tables(conn):
    return query(conn, f"SELECT * FROM {CATALOG_TABLES} ORDER BY table_name")


def catalog_columns(conn, table_name):
    return query(conn, f"SELECT * FROM {CATALOG_COLUMNS} WHERE table_name = ? ORDER BY position", (table_name,))


def describe_table(conn, table_name):
    """Plain-text schema description: what the LLM will see instead of the data."""
    info = conn.execute(f"SELECT * FROM {CATALOG_TABLES} WHERE table_name = ?", (table_name,)).fetchone()
    if info is None:
        raise IngestionError(f"Unknown table '{table_name}'")
    lines = [f"Table {table_name}: {info['row_count']} rows, from '{info['source_file']}' "
             f"(imported {info['imported_at']} UTC)"]
    for col in catalog_columns(conn, table_name).itertuples():
        line = f"- {col.column_name} {col.data_type}"
        if col.original_name and col.original_name != col.column_name:
            line += f" (header: '{col.original_name}')"
        if col.null_count:
            line += f", {col.null_count} blank"
        if pd.notna(col.min_value):
            line += f", range {col.min_value}..{col.max_value}"
        if col.data_type == "text" and col.distinct_count <= CATEGORY_LIMIT:
            name = quote_identifier(col.column_name)
            values = [r[0] for r in conn.execute(
                f"SELECT DISTINCT {name} FROM {quote_identifier(table_name)} WHERE {name} IS NOT NULL ORDER BY 1")]
            line += f", values: {', '.join(repr(v) for v in values)}"
        elif col.data_type == "text":
            line += f", free text (avg {col.avg_length:.0f} chars)"
        lines.append(line)
    return "\n".join(lines)


### Check: table + catalog in one transaction (test database)


In [140]:
conn = connect_db(test_db)
with transaction(conn):
    ensure_catalog(conn)
    written = write_table(conn, "reviews", review_table_columns, review_rows, replace=True)
    record_import(conn, "reviews", raw_file, reviews_sheet.sheet_name, written,
                  review_profiles, reviews_sheet.warnings)

catalog_tables(conn).T


,0
table_name,reviews
source_file,Womens Clothing E-Commerce Reviews.csv
source_path,D:\ChatBot test\data\raw\Womens Clothing E-Com...
source_sheet,Womens Clothing E-Commerce Reviews
source_modified_at,2026-10-14 06:47:36
imported_at,2026-10-14 09:06:23
row_count,23486
warnings,[]


In [141]:
display(catalog_columns(conn, "reviews")[["position", "column_name", "original_name", "data_type",
                                         "null_count", "distinct_count", "sample_values"]])
print(describe_table(conn, "reviews"))


,position,column_name,original_name,data_type,null_count,distinct_count,sample_values
0,1,review_id,,integer,0,23486,"[""0"", ""1"", ""2""]"
1,2,clothing_id,Clothing ID,integer,0,1206,"[""767"", ""1080"", ""1077""]"
2,3,age,Age,integer,0,77,"[""33"", ""34"", ""60""]"
3,4,title,Title,text,3810,13993,"[""Some major design flaws"", ""My favorite buy!""..."
4,5,review_text,Review Text,text,845,22634,"[""Absolutely wonderful - silky and sexy and co..."
5,6,rating,Rating,integer,0,5,"[""4"", ""5"", ""3""]"
6,7,recommended_ind,Recommended IND,integer,0,2,"[""1"", ""0""]"
7,8,positive_feedback_count,Positive Feedback Count,integer,0,82,"[""0"", ""4"", ""6""]"
8,9,division_name,Division Name,text,14,3,"[""Initmates"", ""General"", ""General Petite""]"
9,10,department_name,Department Name,text,14,6,"[""Intimate"", ""Dresses"", ""Bottoms""]"


Table reviews: 23486 rows, from 'Womens Clothing E-Commerce Reviews.csv' (imported 2026-10-14 09:06:23 UTC)
- review_id integer, range 0..23485
- clothing_id integer (header: 'Clothing ID'), range 0..1205
- age integer (header: 'Age'), range 18..99
- title text (header: 'Title'), 3810 blank, free text (avg 19 chars)
- review_text text (header: 'Review Text'), 845 blank, free text (avg 309 chars)
- rating integer (header: 'Rating'), range 1..5
- recommended_ind integer (header: 'Recommended IND'), range 0..1
- positive_feedback_count integer (header: 'Positive Feedback Count'), range 0..122
- division_name text (header: 'Division Name'), 14 blank, values: 'General', 'General Petite', 'Initmates'
- department_name text (header: 'Department Name'), 14 blank, values: 'Bottoms', 'Dresses', 'Intimate', 'Jackets', 'Tops', 'Trend'
- class_name text (header: 'Class Name'), 14 blank, values: 'Blouses', 'Casual bottoms', 'Chemises', 'Dresses', 'Fine gauge', 'Intimates', 'Jackets', 'Jeans', 'Knits

In [142]:
before = catalog_tables(conn)
try:
    with transaction(conn):
        write_table(conn, "reviews", review_table_columns, review_rows[:5] + [(1, 2)], replace=True)
        record_import(conn, "reviews", raw_file, reviews_sheet.sheet_name, 6, review_profiles, [])
    print("UNEXPECTED: no error")
except sqlite3.Error as exc:
    print("Failed as expected ->", exc)

after = catalog_tables(conn)
assert after["imported_at"][0] == before["imported_at"][0]
assert after["row_count"][0] == 23486
print("Catalog unchanged after failed import:", after["row_count"][0], "rows recorded")

conn.close()


Failed as expected -> Incorrect number of bindings supplied. The current statement uses 12, and there are 2 supplied.
Catalog unchanged after failed import: 23486 rows recorded


## Section 8 — `import_file()`
One call: validate → choose sheets → name tables → read → clean names → infer types →
write table + catalog in one transaction. Existing tables are **skipped** unless `replace=True`;
one bad sheet doesn't stop the others.


In [144]:
class TableExistsError(IngestionError):
    """The target table already exists and replace=False."""


@dataclass
class ImportResult:
    sheet: str
    table: str
    status: str                 # "imported", "skipped" or "failed"
    rows: int = 0
    warnings: list = field(default_factory=list)
    message: str = ""


def table_name_for(path, sheet_name, prefix=None):
    """Sheet 'Online Retail' -> online_retail; a CSV is named after its file."""
    path = Path(path)
    base = path.stem if path.suffix.lower() == ".csv" else sheet_name
    if prefix:
        base = f"{prefix}_{base}"
    return normalize_name(base, fallback="sheet")


In [145]:
def import_sheet(conn, path, sheet_name, table, replace=False, renames=None):
    """Read, profile and write one sheet + its catalog entry in a single transaction."""
    renames = renames or {}
    if table_exists(conn, table) and not replace:
        raise TableExistsError(f"Table '{table}' already exists. Use replace=True to overwrite it.")

    sheet = read_sheet(path, sheet_name)
    names = apply_renames(normalize_column_names(sheet.headers), renames)
    warnings = list(sheet.warnings)
    profiles, stored = [], []
    for i, (header, name) in enumerate(zip(sheet.headers, names)):
        typed, profile = profile_column(sheet.data[i], header, name)
        profiles.append(profile)
        stored.append(to_storage_values(typed, profile.data_type))
        if not header:
            warnings.append(f"Column {i + 1} has no header; named it '{name}'")
        elif name != normalize_name(header, fallback=name) and name not in renames.values():
            warnings.append(f"Duplicate column name '{header}' renamed to '{name}'")
        warnings.extend(f"Column '{header or name}': {w}" for w in profile.warnings)

    columns = [(ROW_ID_COLUMN, "integer")] + [(p.name, p.data_type) for p in profiles]
    rows = zip(sheet.data.index.tolist(), *stored)
    with transaction(conn):
        ensure_catalog(conn)
        count = write_table(conn, table, columns, rows, replace=replace)
        record_import(conn, table, path, sheet_name, count, profiles, warnings)
    return ImportResult(sheet_name, table, "imported", rows=count, warnings=warnings)


In [146]:
def import_file(path, db_path=DB_PATH, sheets=None, replace=False, table_prefix=None,
                table_names=None, column_renames=None, max_size_mb=MAX_FILE_SIZE_MB):
    """Import the chosen sheets (default: all) into SQLite. Returns one ImportResult per sheet.

    table_names:    {sheet: table name}      e.g. {"Womens Clothing E-Commerce Reviews": "reviews"}
    column_renames: {sheet: {old: new}}      e.g. {"...": {"column_1": "review_id"}}
    """
    path = validate_file(path, max_size_mb)
    available = list_sheets(path)
    selected = list(dict.fromkeys(sheets)) if sheets else available
    unknown = [s for s in selected if s not in available]
    if unknown:
        raise IngestionError(f"Sheet(s) not found in '{path.name}': {unknown}. Available: {available}")

    table_names = table_names or {}
    column_renames = column_renames or {}
    tables = dedupe_names([
        normalize_name(table_names[s], fallback="sheet") if s in table_names
        else table_name_for(path, s, table_prefix)
        for s in selected
    ])

    results = []
    conn = connect_db(db_path)
    try:
        for sheet_name, table in zip(selected, tables):
            try:
                results.append(import_sheet(conn, path, sheet_name, table, replace,
                                            column_renames.get(sheet_name)))
            except TableExistsError as exc:
                results.append(ImportResult(sheet_name, table, "skipped", message=str(exc)))
            except IngestionError as exc:
                results.append(ImportResult(sheet_name, table, "failed", message=str(exc)))
    finally:
        conn.close()
    return results


def show_results(results):
    for r in results:
        print(f"[{r.status}] '{r.sheet}' -> {r.table}" + (f" ({r.rows} rows)" if r.rows else ""))
        if r.message:
            print("   ", r.message)
        for w in r.warnings:
            print("    warning:", w)


### Check: a two-sheet workbook (import, skip, replace, join)


In [147]:
shop_file = TEST_DIR / "shop.xlsx"
wb = Workbook()
products = wb.active
products.title = "Products"
products.append(["Product ID", "Product Name", "Category"])
products.append([1, "Silk Dress", "Dresses"])
products.append([2, "Wool Sweater", "Tops"])
orders = wb.create_sheet("Orders")
orders.append(["Order ID", "Product ID", "Qty", "Order Date"])
orders.append([101, 1, 2, datetime(2024, 1, 15)])
orders.append([102, 2, 1, datetime(2024, 2, 1)])
orders.append([103, 1, 3, datetime(2024, 2, 3)])
wb.save(shop_file)

shop_db = TEST_DIR / "shop.db"
shop_db.unlink(missing_ok=True)
show_results(import_file(shop_file, db_path=shop_db))
print("--- again without replace")
show_results(import_file(shop_file, db_path=shop_db))
print("--- only Orders, with replace")
show_results(import_file(shop_file, db_path=shop_db, sheets=["Orders"], replace=True))
try:
    import_file(shop_file, db_path=shop_db, sheets=["Customers"])
except IngestionError as exc:
    print("Rejected as expected ->", exc)

conn = connect_db(shop_db, read_only=True)
display(query(conn, """
    SELECT p.category, SUM(o.qty) AS units
    FROM orders o JOIN products p ON p.product_id = o.product_id
    GROUP BY p.category ORDER BY units DESC"""))
display(catalog_tables(conn)[["table_name", "source_sheet", "row_count"]])
conn.close()


[imported] 'Products' -> products (2 rows)
[imported] 'Orders' -> orders (3 rows)
--- again without replace
[skipped] 'Products' -> products
    Table 'products' already exists. Use replace=True to overwrite it.
[skipped] 'Orders' -> orders
    Table 'orders' already exists. Use replace=True to overwrite it.
--- only Orders, with replace
[imported] 'Orders' -> orders (3 rows)
Rejected as expected -> Sheet(s) not found in 'shop.xlsx': ['Customers']. Available: ['Products', 'Orders']


,category,units
0,Dresses,5
1,Tops,1


,table_name,source_sheet,row_count
0,orders,Orders,3
1,products,Products,2


### The real import → `data/processed/chatbot.db`
This is the database every later notebook uses (read-only).


In [148]:
reviews_sheet_name = list_sheets(raw_file)[0]
results = import_file(
    raw_file,
    db_path=DB_PATH,
    replace=True,
    table_names={reviews_sheet_name: "reviews"},
    column_renames={reviews_sheet_name: COLUMN_RENAMES},
)
show_results(results)

conn = connect_db(DB_PATH, read_only=True)
print()
print(describe_table(conn, "reviews"))
conn.close()


[imported] 'Womens Clothing E-Commerce Reviews' -> reviews (23486 rows)

Table reviews: 23486 rows, from 'Womens Clothing E-Commerce Reviews.csv' (imported 2026-10-14 09:19:04 UTC)
- review_id integer, range 0..23485
- clothing_id integer (header: 'Clothing ID'), range 0..1205
- age integer (header: 'Age'), range 18..99
- title text (header: 'Title'), 3810 blank, free text (avg 19 chars)
- review_text text (header: 'Review Text'), 845 blank, free text (avg 309 chars)
- rating integer (header: 'Rating'), range 1..5
- recommended_ind integer (header: 'Recommended IND'), range 0..1
- positive_feedback_count integer (header: 'Positive Feedback Count'), range 0..122
- division_name text (header: 'Division Name'), 14 blank, values: 'General', 'General Petite', 'Initmates'
- department_name text (header: 'Department Name'), 14 blank, values: 'Bottoms', 'Dresses', 'Intimate', 'Jackets', 'Tops', 'Trend'
- class_name text (header: 'Class Name'), 14 blank, values: 'Blouses', 'Casual bottoms', 'Ch

## Section 9 — Final checks on the real database
Compare `chatbot.db` with the CSV read **independently** by plain `pd.read_csv`.
If both routes agree, the import is trustworthy.


In [149]:
expected = pd.read_csv(raw_file)                 # independent reading of the source
conn = connect_db(DB_PATH, read_only=True)
print("Tables in database:", [t for t in query(conn, "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")["name"]])


Tables in database: ['_catalog_columns', '_catalog_tables', 'reviews']


In [150]:
# 1. Same number of rows
db_rows = conn.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
assert db_rows == len(expected), (db_rows, len(expected))

# 2. review_id is exactly 0..n-1 (the key that links SQLite to Weaviate)
ids = query(conn, "SELECT review_id FROM reviews ORDER BY review_id")["review_id"]
assert (ids.values == range(len(expected))).all()

# 3. Same rating distribution as the source
db_ratings = query(conn, "SELECT rating, COUNT(*) AS n FROM reviews GROUP BY rating").set_index("rating")["n"]
assert db_ratings.to_dict() == expected["Rating"].value_counts().to_dict()

# 4. Same blanks
blanks = query(conn, """SELECT SUM(title IS NULL) AS title, SUM(review_text IS NULL) AS review_text,
                               SUM(department_name IS NULL) AS department FROM reviews""").iloc[0]
assert blanks["title"] == expected["Title"].isna().sum()
assert blanks["review_text"] == expected["Review Text"].isna().sum()
assert blanks["department"] == expected["Department Name"].isna().sum()

# 5. Every review text survived exactly (apart from trimmed spaces)
db_text = query(conn, "SELECT review_text FROM reviews ORDER BY review_id")["review_text"].fillna("")
assert (db_text == expected["Review Text"].fillna("").str.strip()).all()

# 6. Numbers are stored as real integers, not text
types = query(conn, "SELECT DISTINCT typeof(rating) AS rating, typeof(age) AS age FROM reviews")
assert types.values.tolist() == [["integer", "integer"]], types

# 7. Freshness: the import is newer than the file
info = catalog_tables(conn).iloc[0]
assert info["source_modified_at"] <= info["imported_at"]

print("All 7 checks passed:", db_rows, "rows; imported", info["imported_at"], "UTC")


All 7 checks passed: 23486 rows; imported 2026-10-14 09:19:04 UTC


In [151]:
row = query(conn, "SELECT * FROM reviews WHERE review_id = 2").iloc[0]
print(f"review_id 2 -> source row {row['_row_id']}: '{row['title']}', {row['rating']} stars, {row['department_name']}")
conn.close()


review_id 2 -> source row 4: 'Some major design flaws', 3 stars, Dresses


## Summary
- `data/processed/chatbot.db` holds table **`reviews`** (23,486 rows, 12 columns incl. `_row_id`)
  plus `_catalog_tables` / `_catalog_columns`.
- Types: 6 integer columns, 5 text columns; no type warnings.
- `review_id` (0..23485) is the key shared with the vector database; `_row_id` is the source row for citations.
- Verified against an independent `pd.read_csv`: row count, IDs, ratings, blanks, every review text.
- Re-run Section 8's last cell whenever the CSV changes (`replace=True`), then rebuild the vector index.
